####  INICIAR NOTEBOOK


In [1]:
%idle_timeout 60
%glue_version 5.1
%worker_type G.1X
%number_of_workers 2

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from awsglue.context import GlueContext
from awsglue.dynamicframe import DynamicFrame
from awsglue.job import Job

from pyspark.context import SparkContext
from pyspark.sql import functions as F

import re
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 60 minutes.
Setting Glue version to: 5.1
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 60
Session ID: 62408d9e-77cd-4d41-b886-26fae8d5d69a
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 62408d9e-77cd-4d41-b886-26fae8d5d69a to get into ready status...
Session 62408d9e-77cd-4d41-b886-26fae8d5d69a has 

#### 1.0 Declarar o bucket e funções necessárias


In [2]:
BUCKET = "fiap26-data-analytics-824232672586"

Função para criar as tabelas fato, consolida as respostas hoje colunadas em linhas para facilitar consulta

In [3]:
def unpivot_padronizado(df, id_col, ano, prefixo, categoria, rotulos, nome_valor="tecnologia"):
    indicadoras = [c for c in df.columns if c.startswith(prefixo) and c[len(prefixo):] in rotulos]
    if not indicadoras:
        print(f"AVISO ({ano}, {categoria}): nenhuma coluna encontrada com prefixo '{prefixo}'")
        return None
    n = len(indicadoras)
    pares = ", ".join(f"'{rotulos[c[len(prefixo):]]}', `{c}`" for c in indicadoras)
    stack_expr = f"stack({n}, {pares}) as ({nome_valor}, utilizado)"
    return (df.selectExpr(id_col, f"cast({ano} as int) as ano_pesquisa", stack_expr)
              .filter("utilizado = '1'")
              .withColumn("categoria", F.lit(categoria))
              .select(F.col(id_col).alias("respondente_id"), "ano_pesquisa", "categoria", nome_valor))

# junta uma lista de resultados de unpivot_padronizado (alguns podem vir
# None quando um prefixo não existir num ano -- ver AVISO) numa fact só
def unir_partes(partes):
    resultado = None
    for parte in partes:
        if parte is None:
            continue
        resultado = parte if resultado is None else resultado.unionByName(parte)
    return resultado

#### 2.0 Ler camada silver


In [4]:
silver_2023 = glueContext.create_dynamic_frame.from_catalog(database="silver_db", table_name="tbl_state_of_data_2023").toDF()
silver_2024 = glueContext.create_dynamic_frame.from_catalog(database="silver_db", table_name="tbl_state_of_data_2024").toDF()
silver_2025 = glueContext.create_dynamic_frame.from_catalog(database="silver_db", table_name="tbl_state_of_data_2025").toDF()

/usr/lib/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.


#### 3.0 Criação das tabelas fato


##### 3.1 Fato Perfil Profissional

In [5]:
CAMPOS_PADRAO = [
    "id", "idade", "faixa_idade", "genero", "cor_raca_etnia", "pcd",
    "uf_atual", "regiao_atual", "nivel_ensino", "setor_atual", "num_funcionarios",
    "atua_como_gestor", "cargo_atual", "senioridade", "faixa_salarial",
    "faixa_salarial_valida", "modalidade_atual", "servico_cloud_preferido",
    "ai_prioridade", "ano_pesquisa",
]

def so_campos_padrao(df, ano):
    cols = [c for c in CAMPOS_PADRAO if c in df.columns]
    faltando = [c for c in CAMPOS_PADRAO if c not in df.columns]
    if faltando:
        print(f"AVISO ({ano}): campos padrao ausentes na Silver: {faltando}")
    sel = df.select(*cols)
    if "id" in cols:
        sel = sel.withColumnRenamed("id", "respondente_id")
    return sel

gold_core = (so_campos_padrao(silver_2023, 2023)
             .unionByName(so_campos_padrao(silver_2024, 2024), allowMissingColumns=True)
             .unionByName(so_campos_padrao(silver_2025, 2025), allowMissingColumns=True))

gold_core.groupBy("ano_pesquisa").count().show()

+------------+-----+
|ano_pesquisa|count|
+------------+-----+
|        2023| 5293|
|        2024| 5215|
|        2025| 3494|
+------------+-----+


In [6]:
dyf_core = DynamicFrame.fromDF(gold_core, glueContext, "gold_core")
sink = glueContext.getSink(
    path=f"s3://{BUCKET}/gold/tbl_fato_perfil_profissional/",
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=["ano_pesquisa"],
    enableUpdateCatalog=True,
)
sink.setCatalogInfo(catalogDatabase="gold_db", catalogTableName="tbl_fato_perfil_profissional")
sink.setFormat("parquet", useGlueParquetWriter=True)
sink.writeFrame(dyf_core)

##### 3.2 Dimensão Faixa Salarial


In [7]:
CORRECOES_FAIXA_SALARIAL = {
    "de R$ 25.001/mês a R$ 3000/mês": "de R$ 25.001/mês a R$ 30000/mês",
}

def parse_faixa_salarial(faixa):
    if faixa is None:
        return (None, None)
    faixa_parse = CORRECOES_FAIXA_SALARIAL.get(faixa, faixa)
    numeros = re.findall(r"[\d.]+", faixa_parse)
    valores = [float(n.replace(".", "")) for n in numeros]
    faixa_lower = faixa_parse.lower()
    if faixa_lower.startswith("menos de"):
        minimo, maximo = (0.0, valores[0]) if valores else (None, None)
    elif faixa_lower.startswith("acima de") or faixa_lower.startswith("mais de"):
        minimo, maximo = (valores[0], None) if valores else (None, None)
    elif len(valores) >= 2:
        minimo, maximo = valores[0], valores[1]
    else:
        minimo, maximo = (None, None)
    if minimo is not None and maximo is not None and minimo > maximo:
        return (None, None)  # valor incoerente na fonte e sem correção conhecida -- não inventa
    return (minimo, maximo)

faixas_distintas = [
    r["faixa_salarial"] for r in gold_core.select("faixa_salarial").distinct().collect()
    if r["faixa_salarial"] is not None
]

linhas_dim = []
for faixa in faixas_distintas:
    minimo, maximo = parse_faixa_salarial(faixa)
    if minimo is None and maximo is None:
        print(f"AVISO (dim_faixa_salarial): nao consegui interpretar a faixa '{faixa}' -- confira manualmente")
    medio = (minimo + maximo) / 2 if (minimo is not None and maximo is not None) else minimo
    corrigida = faixa in CORRECOES_FAIXA_SALARIAL
    linhas_dim.append((faixa, minimo, maximo, medio, corrigida))

dim_faixa_salarial = spark.createDataFrame(
    linhas_dim, ["faixa_salarial", "valor_min", "valor_max", "valor_medio", "valor_corrigido_da_fonte"]
)
dim_faixa_salarial.orderBy("valor_min").show(20, truncate=False)

+--------------------------------+---------+---------+-----------+------------------------+
|faixa_salarial                  |valor_min|valor_max|valor_medio|valor_corrigido_da_fonte|
+--------------------------------+---------+---------+-----------+------------------------+
|Menos de R$ 1.000/mês           |0.0      |1000.0   |500.0      |false                   |
|de R$ 101/mês a R$ 2.000/mês    |101.0    |2000.0   |1050.5     |false                   |
|de R$ 1.001/mês a R$ 2.000/mês  |1001.0   |2000.0   |1500.5     |false                   |
|de R$ 2.001/mês a R$ 3.000/mês  |2001.0   |3000.0   |2500.5     |false                   |
|de R$ 3.001/mês a R$ 4.000/mês  |3001.0   |4000.0   |3500.5     |false                   |
|de R$ 4.001/mês a R$ 6.000/mês  |4001.0   |6000.0   |5000.5     |false                   |
|de R$ 6.001/mês a R$ 8.000/mês  |6001.0   |8000.0   |7000.5     |false                   |
|de R$ 8.001/mês a R$ 12.000/mês |8001.0   |12000.0  |10000.5    |false         

In [8]:
dyf_dim_faixa = DynamicFrame.fromDF(dim_faixa_salarial, glueContext, "dim_faixa_salarial")
sink = glueContext.getSink(
    path=f"s3://{BUCKET}/gold/tbl_dimensao_faixa_salarial/",
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    enableUpdateCatalog=True,
)
sink.setCatalogInfo(catalogDatabase="gold_db", catalogTableName="tbl_dimensao_faixa_salarial")
sink.setFormat("parquet", useGlueParquetWriter=True)
sink.writeFrame(dyf_dim_faixa)

##### 3.3 Fato Linguagens Programação

In [25]:
ROTULOS_LINGUAGEM = {
    "sql": "SQL", "r": "R", "python": "Python", "c": "C/C++/C#", "net": ".NET",
    "java": "Java", "julia": "Julia", "sas": "SAS/Stata", "vba": "Visual Basic/VBA",
    "scala": "Scala", "matlab": "Matlab", "rust": "Rust", "php": "PHP",
    "javascript": "JavaScript", "dax": "DAX",
}
ROTULOS_BANCO_DADOS = {
    "mysql": "MySQL", "oracle": "Oracle", "sqlserver": "SQL Server", "rds": "Amazon RDS",
    "dynamodb": "DynamoDB", "coachdb": "CouchDB", "cassandra": "Cassandra", "mongodb": "MongoDB",
    "mariadb": "MariaDB", "datomic": "Datomic", "s3": "Amazon S3", "postgresql": "PostgreSQL",
    "elasticsearch": "Elasticsearch", "db2": "DB2", "microsoftacess": "Microsoft Access",
    "sqlite": "SQLite", "sybase": "Sybase", "firebase": "Firebase", "vertica": "Vertica",
    "redis": "Redis", "neo4j": "Neo4J", "bigquery": "Google BigQuery",
    "firestone": "Google Firestore", "redshift": "Amazon Redshift", "athena": "Amazon Athena",
    "snowflake": "Snowflake", "databricks": "Databricks", "hbase": "HBase", "presto": "Presto",
    "splunk": "Splunk", "saphana": "SAP HANA", "hive": "Hive", "firebird": "Firebird",
}
ROTULOS_FERRAMENTA_BI = {
    "powerbi": "Microsoft Power BI", "qlik": "Qlik View/Sense", "tableau": "Tableau",
    "metabase": "Metabase", "superset": "Superset", "redash": "Redash", "looker": "Looker",
    "lookerstudio": "Looker Studio", "quicksight": "Amazon QuickSight", "alteryx": "Alteryx",
    "sap": "SAP Business Objects", "oracle": "Oracle BI", "salesforce": "Salesforce/Einstein Analytics",
    "sas": "SAS Visual Analytics", "grafana": "Grafana", "pentaho": "Pentaho",
    "excel": "Excel/Google Sheets",
}

ROTULOS_TECNICAS_DS = {
    "regressao": "Modelos de regressão", "redes_neurais": "Redes neurais/árvore",
    "recomendacao": "Sistemas de recomendação", "bayesianos": "Métodos Bayesianos",
    "nlp": "NLP", "estatistica": "Estatística clássica", "markov": "Cadeias de Markov/HMM",
    "clusterizacao": "Clusterização", "series_temporais": "Séries temporais",
    "reforco": "Reinforcement Learning", "ml": "ML para detecção de fraude",
    "visao_computacional": "Visão computacional", "churn": "Detecção de churn",
    "llm": "LLMs para negócio",
}

ROTULOS_FERRAMENTA_DS = {
    "bi": "Ferramentas de BI", "planilhas": "Planilhas", "local": "Dev local (RStudio/Jupyter)",
    "nuvem": "Dev na nuvem (Colab/SageMaker)", "automl": "AutoML", "etl": "Ferramentas de ETL",
    "ml": "Plataformas de ML", "feature_store": "Feature Store",
    "controle_versao": "Controle de versão", "data_apps": "Data Apps (Streamlit etc.)",
    "estatistica": "Estatística avançada",
}

ROTULOS_FERRAMENTA_ETL = {
    "python": "Scripts Python", "sql": "SQL & Stored Procedures", "airflow": "Apache Airflow",
    "nifi": "Apache NiFi", "luigi": "Luigi", "glue": "AWS Glue", "talend": "Talend",
    "pentaho": "Pentaho", "alteryx": "Alteryx", "stitch": "Stitch", "fivetran": "Fivetran",
    "dataflow": "Google Dataflow", "oracle": "Oracle Data Integrator", "ibm": "IBM DataStage",
    "sap": "SAP BW ETL", "sqlserver": "SSIS", "sas": "SAS Data Integration",
    "qliksense": "Qlik Sense", "knime": "Knime", "databricks": "Databricks",
}
ROTULOS_FERRAMENTA_NEGOCIOS = {
    "automl": "AutoML", "point_click": "Point-and-Click Analytics",
    "product_metrics": "Product Metrics & Insights", "crm": "Análise dentro de CRM",
    "sem_uso": "Empresa não usa essas ferramentas", "sem_opiniao": "Não sei informar",
}
ROTULOS_TIPO_DADO = {
    "relacionais": "Dados relacionais (SQL)", "nosql": "Dados em bancos NoSQL",
    "imagens": "Imagens", "documentos": "Textos/Documentos", "videos": "Vídeos",
    "audios": "Áudios", "planilhas": "Planilhas", "georeferenciados": "Dados georeferenciados",
}

ling_2023 = unpivot_padronizado(silver_2023, "id", 2023, "linguagem_programacao_", "linguagem", ROTULOS_LINGUAGEM)
ling_2024 = unpivot_padronizado(silver_2024, "id", 2024, "linguagem_programacao_", "linguagem", ROTULOS_LINGUAGEM)
ling_2025 = unpivot_padronizado(silver_2025, "id", 2025, "linguagem_programacao_", "linguagem", ROTULOS_LINGUAGEM)

partes_tecnologias = [ling_2023, ling_2024, ling_2025]
for ano, silver_ano in [(2023, silver_2023), (2024, silver_2024), (2025, silver_2025)]:
    partes_tecnologias.append(unpivot_padronizado(silver_ano, "id", ano, "bancos_dados_", "banco_dados", ROTULOS_BANCO_DADOS))
    partes_tecnologias.append(unpivot_padronizado(silver_ano, "id", ano, "ferramenta_bi_", "ferramenta_bi", ROTULOS_FERRAMENTA_BI))
    partes_tecnologias.append(unpivot_padronizado(silver_ano, "id", ano, "tecnicas_ds_", "tecnica_cientista", ROTULOS_TECNICAS_DS))
    partes_tecnologias.append(unpivot_padronizado(silver_ano, "id", ano, "ferramenta_ds_", "ferramenta_cientista", ROTULOS_FERRAMENTA_DS))
    partes_tecnologias.append(unpivot_padronizado(silver_ano, "id", ano, "ferramenta_etl_", "ferramenta_etl_engenheiro", ROTULOS_FERRAMENTA_ETL))
    partes_tecnologias.append(unpivot_padronizado(silver_ano, "id", ano, "ferramenta_etl_da_", "ferramenta_etl_analista", ROTULOS_FERRAMENTA_ETL))
    partes_tecnologias.append(unpivot_padronizado(silver_ano, "id", ano, "ferramenta_negocios_", "ferramenta_negocios", ROTULOS_FERRAMENTA_NEGOCIOS))
    partes_tecnologias.append(unpivot_padronizado(silver_ano, "id", ano, "origem_dados_usados_", "tipo_dado_mais_usado", ROTULOS_TIPO_DADO))
    partes_tecnologias.append(unpivot_padronizado(silver_ano, "id", ano, "origem_dados_", "tipo_dado_trabalhado", ROTULOS_TIPO_DADO))

fact_tecnologias = unir_partes(partes_tecnologias)
fact_tecnologias.groupBy("categoria", "tecnologia").count().orderBy(F.desc("count")).show(20)

AVISO (2025, tipo_dado_mais_usado): nenhuma coluna encontrada com prefixo 'origem_dados_usados_'
+--------------------+--------------------+-----+
|           categoria|          tecnologia|count|
+--------------------+--------------------+-----+
|tipo_dado_trabalhado|Dados relacionais...| 8726|
|tipo_dado_trabalhado|           Planilhas| 8597|
|           linguagem|                 SQL| 8064|
|           linguagem|              Python| 7687|
|tipo_dado_mais_usado|Dados relacionais...| 6021|
|tipo_dado_trabalhado|   Textos/Documentos| 5620|
|       ferramenta_bi|  Microsoft Power BI| 5392|
|tipo_dado_mais_usado|           Planilhas| 4081|
|tipo_dado_trabalhado|Dados em bancos N...| 3577|
|         banco_dados|          PostgreSQL| 2965|
|         banco_dados|          SQL Server| 2963|
|tipo_dado_trabalhado|Dados georeferenc...| 2934|
|         banco_dados|          Databricks| 2612|
|         banco_dados|     Google BigQuery| 2490|
|         banco_dados|               MySQL| 2357|
|  

In [26]:
dyf_tec = DynamicFrame.fromDF(fact_tecnologias, glueContext, "gold_tecnologias")
sink = glueContext.getSink(
    path=f"s3://{BUCKET}/gold/tbl_fato_tecnologias/",
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=["ano_pesquisa"],
    enableUpdateCatalog=True,
)
sink.setCatalogInfo(catalogDatabase="gold_db", catalogTableName="tbl_fato_tecnologias")
sink.setFormat("parquet", useGlueParquetWriter=True)
sink.writeFrame(dyf_tec)

##### 3.4 Fato Experiência

In [12]:
ROTULOS_ASPECTOS_PREJUDICADOS = {
    "qtd_vagas": "Quantidade de vagas recebidas",
    "senioridade_vagas": "Senioridade das vagas recebidas",
    "aprovacao_vagas": "Aprovação em processos seletivos",
    "progressao_vagas": "Oportunidades de progressão",
    "velocidade_progressao": "Velocidade de progressão",
    "nivel_stress": "Nível de cobrança/stress",
    "nivel_visibilidade": "Atenção às minhas opiniões/ideias",
    "network_interno": "Relação no trabalho (interno)",
    "network_externo": "Relação em momentos de integração",
}

partes_prejudicada = []
for ano, silver_ano in [(2023, silver_2023), (2024, silver_2024), (2025, silver_2025)]:
    partes_prejudicada.append(unpivot_padronizado(
        silver_ano, "id", ano, "aspectos_prejudicados_", "aspecto_prejudicado",
        ROTULOS_ASPECTOS_PREJUDICADOS, nome_valor="item"
    ))
fact_experiencia_prejudicada = unir_partes(partes_prejudicada)
fact_experiencia_prejudicada.groupBy("item").count().orderBy(F.desc("count")).show(20, truncate=False)

+---------------------------------+-----+
|item                             |count|
+---------------------------------+-----+
|Atenção às minhas opiniões/ideias|1836 |
|Velocidade de progressão         |1792 |
|Oportunidades de progressão      |1727 |
|Aprovação em processos seletivos |1290 |
|Quantidade de vagas recebidas    |1289 |
|Senioridade das vagas recebidas  |1191 |
|Relação no trabalho (interno)    |1035 |
|Nível de cobrança/stress         |941  |
|Relação em momentos de integração|842  |
+---------------------------------+-----+


In [13]:
dyf_prej = DynamicFrame.fromDF(fact_experiencia_prejudicada, glueContext, "gold_prejudicada")
sink = glueContext.getSink(
    path=f"s3://{BUCKET}/gold/tbl_fato_experiencia/",
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=["ano_pesquisa"],
    enableUpdateCatalog=True,
)
sink.setCatalogInfo(catalogDatabase="gold_db", catalogTableName="tbl_fato_experiencia")
sink.setFormat("parquet", useGlueParquetWriter=True)
sink.writeFrame(dyf_prej)

##### 3.5 Fato IA Generativa

In [14]:
ROTULOS_AI_USO = {
    "independente": "Uso independente/descentralizado",
    "centralizado": "Direcionamento centralizado",
    "copilotos": "Devs usando Copilots",
    "produtos_externos": "Melhorar produtos externos",
    "produtos_internos": "Melhorar produtos internos",
    "principal": "Principal frente do negócio",
    "sem_uso": "Não é prioridade",
    "sem_opiniao": "Não sei opinar",
}
ROTULOS_AI_USO_TRABALHO = {
    "sem_uso": "Não uso IA para produtividade",
    "gratuito": "Uso solução gratuita",
    "pago": "Uso e pago (próprio bolso)",
    "empresa_paga": "Empresa paga a solução",
    "copilot": "Uso solução tipo Copilot",
}
ROTULOS_AI_MOTIVO_DESUSO = {
    "cases": "Falta de casos de uso claros",
    "confiabilidade": "Falta de confiabilidade (alucinação)",
    "regulamentacao": "Incerteza regulatória",
    "seguranca": "Preocupação com segurança/privacidade",
    "roi": "ROI não comprovado",
    "dados": "Dados não prontos pra IA",
    "expertise": "Falta de expertise/recursos",
    "lideranca": "Liderança não vê valor",
    "propriedade_intelectual": "Preocupação com propriedade intelectual",
}

partes_ia = []
for ano, silver_ano in [(2023, silver_2023), (2024, silver_2024), (2025, silver_2025)]:
    partes_ia.append(unpivot_padronizado(silver_ano, "id", ano, "ai_uso_v2_", "uso_empresa_gestor", ROTULOS_AI_USO, nome_valor="item"))
    partes_ia.append(unpivot_padronizado(silver_ano, "id", ano, "ai_uso_trabalho_", "uso_pessoal", ROTULOS_AI_USO_TRABALHO, nome_valor="item"))
    partes_ia.append(unpivot_padronizado(silver_ano, "id", ano, "ai_motivo_desuso_", "motivo_desuso", ROTULOS_AI_MOTIVO_DESUSO, nome_valor="item"))
    partes_ia.append(unpivot_padronizado(silver_ano, "id", ano, "ai_uso_", "uso_empresa", ROTULOS_AI_USO, nome_valor="item"))
fact_ia_generativa = unir_partes(partes_ia)
fact_ia_generativa.groupBy("categoria", "item").count().orderBy(F.desc("count")).show(30, truncate=False)

+------------------+---------------------------------------+-----+
|categoria         |item                                   |count|
+------------------+---------------------------------------+-----+
|uso_pessoal       |Uso solução gratuita                   |4987 |
|uso_empresa_gestor|Uso independente/descentralizado       |4550 |
|uso_empresa_gestor|Melhorar produtos externos             |2450 |
|uso_empresa_gestor|Melhorar produtos internos             |2234 |
|uso_empresa_gestor|Devs usando Copilots                   |2209 |
|uso_pessoal       |Uso solução tipo Copilot               |1930 |
|uso_empresa_gestor|Direcionamento centralizado            |1911 |
|uso_pessoal       |Empresa paga a solução                 |1828 |
|uso_pessoal       |Uso e pago (próprio bolso)             |1390 |
|uso_empresa_gestor|Não é prioridade                       |1336 |
|uso_pessoal       |Não uso IA para produtividade          |1024 |
|uso_empresa       |Uso independente/descentralizado       |10

In [15]:
dyf_ia = DynamicFrame.fromDF(fact_ia_generativa, glueContext, "gold_ia_generativa")
sink = glueContext.getSink(
    path=f"s3://{BUCKET}/gold/tbl_fato_ia_generativa/",
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=["ano_pesquisa"],
    enableUpdateCatalog=True,
)
sink.setCatalogInfo(catalogDatabase="gold_db", catalogTableName="tbl_fato_ia_generativa")
sink.setFormat("parquet", useGlueParquetWriter=True)
sink.writeFrame(dyf_ia)

##### 3.6 Fato Gestão

In [20]:
ROTULOS_DESAFIOS_GESTOR = {
    "contratar": "Contratar talentos", "reter": "Reter talentos",
    "investir": "Convencer a aumentar investimento", "remoto": "Gestão de equipes remotas",
    "multidisciplinaridade": "Projetos multidisciplinares", "qualidade": "Qualidade/confiabilidade dos dados",
    "dados": "Processar/armazenar alto volume", "valor": "Gerar valor pro negócio",
    "modelos": "Manter modelos de ML em produção", "expectativa": "Gerenciar expectativa do negócio",
    "manutencao": "Manter projetos em produção", "inovacao": "Levar inovação pra empresa",
    "roi": "Garantir ROI dos projetos", "tempo": "Dividir tempo técnico/gestão",
}
ROTULOS_RESPONSABILIDADES_GESTOR = {
    "lp_dados": "Visão de longo prazo de dados", "treinamentos": "Treinamentos/maturidade analítica",
    "selecao": "Atração/seleção de talentos", "ferramentas": "Contratação de ferramentas",
    "equipe_engenharia": "Gestor de engenharia de dados", "equipe_estudos": "Gestor de estudos/relatórios",
    "equipe_mlai": "Gestor de IA/ML", "tecnico": "Ainda atua tecnicamente",
    "projetos_dados": "Gestão de projetos de dados", "produtos_dados": "Gestão de produtos de dados",
    "pessoas": "Gestão de pessoas",
}
partes_gestao = []
for ano, silver_ano in [(2023, silver_2023), (2024, silver_2024), (2025, silver_2025)]:
    partes_gestao.append(unpivot_padronizado(silver_ano, "id", ano, "desafios_como_gestor_", "desafio", ROTULOS_DESAFIOS_GESTOR, nome_valor="item"))
    partes_gestao.append(unpivot_padronizado(silver_ano, "id", ano, "responsabilidades_como_gestor_", "responsabilidade", ROTULOS_RESPONSABILIDADES_GESTOR, nome_valor="item"))
fact_gestao = unir_partes(partes_gestao)
fact_gestao.groupBy("categoria", "item").count().orderBy(F.desc("count")).show(20, truncate=False)

+----------------+----------------------------------+-----+
|categoria       |item                              |count|
+----------------+----------------------------------+-----+
|responsabilidade|Visão de longo prazo de dados     |1638 |
|responsabilidade|Gestão de pessoas                 |1525 |
|responsabilidade|Gestor de estudos/relatórios      |1319 |
|responsabilidade|Atração/seleção de talentos       |1207 |
|responsabilidade|Gestão de projetos de dados       |1152 |
|responsabilidade|Treinamentos/maturidade analítica |1023 |
|responsabilidade|Ainda atua tecnicamente           |990  |
|responsabilidade|Contratação de ferramentas        |966  |
|responsabilidade|Gestão de produtos de dados       |915  |
|desafio         |Gerenciar expectativa do negócio  |902  |
|responsabilidade|Gestor de IA/ML                   |873  |
|desafio         |Dividir tempo técnico/gestão      |813  |
|responsabilidade|Gestor de engenharia de dados     |812  |
|desafio         |Gerar valor pro negóci

In [21]:
dyf_gestao = DynamicFrame.fromDF(fact_gestao, glueContext, "gold_gestao")
sink = glueContext.getSink(
    path=f"s3://{BUCKET}/gold/tbl_fato_gestao/",
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=["ano_pesquisa"],
    enableUpdateCatalog=True,
)
sink.setCatalogInfo(catalogDatabase="gold_db", catalogTableName="tbl_fato_gestao")
sink.setFormat("parquet", useGlueParquetWriter=True)
sink.writeFrame(dyf_gestao)

##### 3.7 Fato Time Dados

In [22]:
ROTULOS_CARGOS_DADOS = {
    "analytics_engineer": "Analytics Engineer", "data_engineer": "Engenharia de Dados",
    "data_analyst": "Analista de Dados", "data_scientist": "Cientista de Dados", "dba": "DBA",
    "bi_analst": "Analista de BI", "data_architect": "Arquiteto de Dados", "dpm": "Data Product Manager",
    "business_analyst": "Business Analyst", "mlai_engineer": "ML/AI Engineer",
}

partes_estrutura = []
for ano, silver_ano in [(2023, silver_2023), (2024, silver_2024), (2025, silver_2025)]:
    partes_estrutura.append(unpivot_padronizado(silver_ano, "id", ano, "cargos_dados_", "cargo_no_time", ROTULOS_CARGOS_DADOS, nome_valor="item"))
fact_estrutura_time_dados = unir_partes(partes_estrutura)
fact_estrutura_time_dados.groupBy("item").count().orderBy(F.desc("count")).show(20, truncate=False)

dyf_estrutura = DynamicFrame.fromDF(fact_estrutura_time_dados, glueContext, "gold_estrutura_time")
sink = glueContext.getSink(
    path=f"s3://{BUCKET}/gold/tbl_fato_time_dados/",
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=["ano_pesquisa"],
    enableUpdateCatalog=True,
)
sink.setCatalogInfo(catalogDatabase="gold_db", catalogTableName="tbl_fato_time_dados")
sink.setFormat("parquet", useGlueParquetWriter=True)
sink.writeFrame(dyf_estrutura)

+--------------------+-----+
|item                |count|
+--------------------+-----+
|Analista de Dados   |1683 |
|Engenharia de Dados |1669 |
|Cientista de Dados  |1539 |
|Analista de BI      |1330 |
|Business Analyst    |949  |
|Analytics Engineer  |848  |
|Arquiteto de Dados  |848  |
|ML/AI Engineer      |628  |
|Data Product Manager|600  |
|DBA                 |529  |
+--------------------+-----+



##### 3.8 Fato Satisfação

In [23]:
ROTULOS_CRITERIOS_EMPREGO = {
    "salario": "Remuneração/Salário", "beneficios": "Benefícios",
    "proposito": "Propósito do trabalho/empresa", "modalidade": "Flexibilidade de trabalho remoto",
    "ambiente": "Ambiente e clima de trabalho", "desenvolvimento": "Aprendizado/referências na área",
    "progressao": "Plano de carreira", "maturidade_dadosti": "Maturidade em tecnologia/dados",
    "relacao_gestao": "Qualidade dos gestores/líderes", "reputacao": "Reputação no mercado",
}
ROTULOS_INSATISFACAO_ATUAL = {
    "salario": "Salário não corresponde ao mercado", "beneficios": "Poucos benefícios",
    "proposito": "Propósito do trabalho/empresa", "modalidade": "Falta de flexibilidade remota",
    "ambiente": "Clima de trabalho ruim", "desenvolvimento": "Falta de aprendizado/referências",
    "progressao": "Falta de oportunidade de crescimento", "maturidade_dadosti": "Falta de maturidade analítica",
    "relacao_gestao": "Relação ruim com líder/gestor", "reputacao": "Reputação no mercado",
    "migrar": "Quer trabalhar em outra área",
}

partes_satisfacao = []
for ano, silver_ano in [(2023, silver_2023), (2024, silver_2024), (2025, silver_2025)]:
    partes_satisfacao.append(unpivot_padronizado(silver_ano, "id", ano, "criterios_emprego_", "criterio_escolha", ROTULOS_CRITERIOS_EMPREGO, nome_valor="item"))
    partes_satisfacao.append(unpivot_padronizado(silver_ano, "id", ano, "insatisfacao_atual_", "motivo_insatisfacao", ROTULOS_INSATISFACAO_ATUAL, nome_valor="item"))
fact_satisfacao_emprego = unir_partes(partes_satisfacao)
fact_satisfacao_emprego.groupBy("categoria", "item").count().orderBy(F.desc("count")).show(30, truncate=False)

dyf_satisfacao = DynamicFrame.fromDF(fact_satisfacao_emprego, glueContext, "gold_satisfacao")
sink = glueContext.getSink(
    path=f"s3://{BUCKET}/gold/tbl_fato_satisfacao/",
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=["ano_pesquisa"],
    enableUpdateCatalog=True,
)
sink.setCatalogInfo(catalogDatabase="gold_db", catalogTableName="tbl_fato_satisfacao")
sink.setFormat("parquet", useGlueParquetWriter=True)
sink.writeFrame(dyf_satisfacao)

+-------------------+------------------------------------+-----+
|categoria          |item                                |count|
+-------------------+------------------------------------+-----+
|criterio_escolha   |Remuneração/Salário                 |10470|
|criterio_escolha   |Flexibilidade de trabalho remoto    |7390 |
|criterio_escolha   |Plano de carreira                   |4172 |
|criterio_escolha   |Aprendizado/referências na área     |3103 |
|criterio_escolha   |Benefícios                          |3101 |
|criterio_escolha   |Ambiente e clima de trabalho        |2599 |
|criterio_escolha   |Propósito do trabalho/empresa       |2025 |
|motivo_insatisfacao|Salário não corresponde ao mercado  |1698 |
|criterio_escolha   |Maturidade em tecnologia/dados      |1574 |
|motivo_insatisfacao|Falta de oportunidade de crescimento|1519 |
|motivo_insatisfacao|Falta de maturidade analítica       |1219 |
|criterio_escolha   |Qualidade dos gestores/líderes      |985  |
|motivo_insatisfacao|Falt

##### 3.9 Fato Rotina

In [24]:
ROTULOS_ROTINA_ENGENHEIRO = {
    "pipelines": "Pipelines de dados via código", "etls": "Construção de ETLs em ferramenta",
    "sql": "Consultas SQL para o negócio", "integracao": "Integração via plataforma proprietária",
    "arquitetura": "Arquitetura de dados", "manutencao": "Data Lake/Lakehouse (streaming)",
    "modelagem": "Modelagem (DW/Data Marts)", "qualidade": "Qualidade de dados/metadados",
    # "nenhuma_listada" excluído -- não é uma atividade
}
ROTULOS_ROTINA_ANALISTA = {
    "analise": "Processamento/análise via código", "dashboard": "Dashboards em ferramenta de BI",
    "consultas": "Consultas SQL para o negócio", "extracao": "Extração via APIs",
    "experimentos": "Experimentos/testes estatísticos", "manutencao": "Manutenção de ETLs",
    "modelagem": "Modelagem (DW/Data Marts)", "planilhas": "Planilhas para o negócio",
    "estatistica": "Estatística avançada (SAS/SPSS)",
}
ROTULOS_ROTINA_CIENTISTA = {
    "estudos": "Estudos ad-hoc/modelos preditivos", "coleta": "Coleta e limpeza de dados",
    "contatos": "Contato com áreas de negócio", "modelagem": "Modelos de ML pra produção",
    "produtizacao": "Produtização (pipelines/APIs)", "manutencao": "Manutenção de modelos em produção",
    "dashboard": "Dashboards em ferramenta de BI", "estatistica": "Estatística avançada",
    "pipeline": "ETLs/DAGs de pipelines", "gerenciamento": "Feature Store/MLOps",
    "infraestrutura": "Infraestrutura (clusters/APIs)", "llm": "Treino/aplicação de LLMs",
}
partes_rotina = []
for ano, silver_ano in [(2023, silver_2023), (2024, silver_2024), (2025, silver_2025)]:
    partes_rotina.append(unpivot_padronizado(silver_ano, "id", ano, "rotina_de_", "rotina_engenheiro", ROTULOS_ROTINA_ENGENHEIRO, nome_valor="item"))
    partes_rotina.append(unpivot_padronizado(silver_ano, "id", ano, "rotina_da_", "rotina_analista", ROTULOS_ROTINA_ANALISTA, nome_valor="item"))
    partes_rotina.append(unpivot_padronizado(silver_ano, "id", ano, "rotina_ds_", "rotina_cientista", ROTULOS_ROTINA_CIENTISTA, nome_valor="item"))
    partes_rotina.append(unpivot_padronizado(silver_ano, "id", ano, "tempo_gasto_", "tempo_engenheiro", ROTULOS_ROTINA_ENGENHEIRO, nome_valor="item"))
    partes_rotina.append(unpivot_padronizado(silver_ano, "id", ano, "tempo_gasto_da_", "tempo_analista", ROTULOS_ROTINA_ANALISTA, nome_valor="item"))
    partes_rotina.append(unpivot_padronizado(silver_ano, "id", ano, "tempo_gasto_ds_", "tempo_cientista", ROTULOS_ROTINA_CIENTISTA, nome_valor="item"))
fact_rotina_trabalho = unir_partes(partes_rotina)
fact_rotina_trabalho.groupBy("categoria").count().show(20, truncate=False)

dyf_rotina = DynamicFrame.fromDF(fact_rotina_trabalho, glueContext, "gold_rotina")
sink = glueContext.getSink(
    path=f"s3://{BUCKET}/gold/tbl_fato_rotina/",
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=["ano_pesquisa"],
    enableUpdateCatalog=True,
)
sink.setCatalogInfo(catalogDatabase="gold_db", catalogTableName="tbl_fato_rotina")
sink.setFormat("parquet", useGlueParquetWriter=True)
sink.writeFrame(dyf_rotina)

+-----------------+-----+
|categoria        |count|
+-----------------+-----+
|rotina_engenheiro|10331|
|rotina_analista  |15462|
|rotina_cientista |8763 |
|tempo_engenheiro |4101 |
|tempo_analista   |7102 |
|tempo_cientista  |3494 |
+-----------------+-----+



##### 3.10 Fato Rotina